<a href="https://colab.research.google.com/github/yantariel/GP_ML_HW/blob/main/GP_Seminars_2026/Seminar_3_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Стратегии генерации


(based on https://github.com/huggingface/blog/blob/main/notebooks/02_how_to_generate.ipynb)

In [1]:
import transformers, torch
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)

transformers: 5.20.2
torch: 2.11.0+cu130


In [2]:
import torch

In [3]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

model = GPT2LMHeadModel.from_pretrained("gpt2", pad_token_id=tokenizer.eos_token_id)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

### **Greedy Search**

Жадный поиск просто выбирает слово с наибольшей вероятностью в качестве следующего слова: $w_t = argmax_{w}P(w | w_{1:t-1})$ на каждом временном шаге $t$. На следующем рисунке показан Greedy Search.

![Greedy Search](https://raw.githubusercontent.com/patrickvonplaten/scientific_images/master/greedy_search.png)

Начиная со слова $\text{"The"}$, алгоритм
жадно выбирает следующее слово с наибольшей вероятностью $\text{"nice"}$ и так далее, так что конечная сгенерированная последовательность слов будет $\text{"The", "nice", "woman"}$ с общей вероятностью $0.5 \times 0.4 = 0.2$.

В дальнейшем мы будем генерировать последовательности слов, используя GPT2 в контексте $(\text{"I", "enjoy", "walking", "with", "my", "cute", "dog"})$. Давайте посмотрим, как можно использовать жадный поиск в "трансформерах" следующим образом
:

In [4]:
# encode context the generation is conditioned on
input_ids = tokenizer.encode('I enjoy walking with my cute dog', return_tensors='pt')

# generate text until the output length (which includes the context length) reaches 50
greedy_output = model.generate(input_ids, max_length=60)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with my dog. I'm not sure if I'll ever be able to walk with my dog.

I'm not sure if I'll ever be able to walk with my dog. I


In [5]:
greedy_output

tensor([[   40,  2883,  6155,   351,   616, 13779,  3290,    11,   475,   314,
          1101,   407,  1654,   611,   314,  1183,  1683,   307,  1498,   284,
          2513,   351,   616,  3290,    13,   314,  1101,   407,  1654,   611,
           314,  1183,  1683,   307,  1498,   284,  2513,   351,   616,  3290,
            13,   198,   198,    40,  1101,   407,  1654,   611,   314,  1183,
          1683,   307,  1498,   284,  2513,   351,   616,  3290,    13,   314]])



Однако основным недостатком жадного поиска является то, что он пропускает слова с высокой вероятностью, скрытые за словом с низкой вероятностью, как видно из нашего наброска выше:

Слово $\text{"has"}$ с его высокой условной вероятностью $0,9$ скрыто за словом $\text{"dog"}$, которое имеет только вторую по величине условную вероятность, так что жадный поиск пропускает последовательность слов $\text{"The"}, \text{"dog"}, \text{"has"}$.

К счастью, у нас есть beam search, чтобы решить эту проблему!


### **Beam search**

Поиск по лучу (beam search) снижает риск пропуска скрытых последовательностей слов с высокой вероятностью, сохраняя наиболее вероятные "num_beams" гипотез на каждом временном шаге и в конечном итоге выбирая гипотезу, имеющую в целом наибольшую вероятность. Давайте проиллюстрируем это с помощью `num_beams=2`:

![Beam search](https://raw.githubusercontent.com/patrickvonplaten/scientific_images/master/beam_search.png)

На временном шаге $1$, помимо наиболее вероятной гипотезы $\text{"The", "woman"}$, beam search также отслеживает вторую наиболее вероятную гипотезу $\text{"The", "dog"}$. На временном шаге $2$ поиск по лучу обнаруживает, что последовательность слов $\text{"The", "dog", "has"}$ имеет вероятность $0,36$ более высокую, чем $\text{"The", "nice", "woman"}$, которая имеет $0,2$. Отлично, он нашел наиболее вероятную последовательность слов в нашем примере!

Поиск по лучу всегда найдет выходную последовательность с большей вероятностью, чем жадный поиск, но не гарантирует нахождения наиболее вероятного результата.

Давайте посмотрим, как поиск по лучу можно использовать в "трансформерах". Мы устанавливаем `num_beams > 1` и `early_stopping=True`, чтобы генерация завершалась, когда все гипотезы луча достигали токена EOS.

In [6]:
# activate beam search and early_stopping
beam_output = model.generate(
    input_ids,
    max_length=60,
    num_beams=5,
    early_stopping=True
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(beam_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again.

I'm not sure if I'll ever be able to walk with him again.

I'm not sure if I'll ever be able to walk with him again


Хотя результат, возможно, более плавный, выход по-прежнему включает повторения одних и тех же последовательностей слов.  
Простое решение состоит в том, чтобы ввести штрафы за *n-граммы*

Давайте попробуем это, установив `no_repeat_n_gram_size=2`, чтобы ни один *2-gramm* не появлялся дважды:

In [7]:
# set no_repeat_ngram_size to 2
beam_output = model.generate(
    input_ids,
    max_length=50,
    num_beams=5,
    no_repeat_ngram_size=2,
    early_stopping=True
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(beam_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again.

I've been thinking about this for a while now, and I think it's time for me to take a break


Это выглядит намного лучше! Мы можем видеть, что повторение больше не появляется. Тем не менее, *n-граммные* штрафы следует использовать с осторожностью. В статье, сгенерированной о городе *Нью-Йорк*, не следует использовать штраф в размере *2-gramms* или название города будет встречаться только один раз во всем тексте!

Еще одной важной особенностью лучевого поиска луча является то, что мы можем сравнить лучшие лучи после генерации и выбрать сгенерированный луч, который лучше всего соответствует нашей цели.

В `transformers` мы просто устанавливаем параметр `num_return_sequences` на количество лучей с наибольшим количеством баллов, которые должны быть возвращены. Убедитесь, однако, что `num_return_sequences <= num_beams`!

In [8]:
# set return_num_sequences > 1
beam_outputs = model.generate(
    input_ids,
    max_length=50,
    num_beams=20,
    no_repeat_ngram_size=4,
    num_return_sequences=5,
    early_stopping=True
)

# now we have 3 output sequences
print("Output:\n" + 100 * '-')
for i, beam_output in enumerate(beam_outputs):
  print("{}: {}\n".format(i, tokenizer.decode(beam_output, skip_special_tokens=True)))

Output:
----------------------------------------------------------------------------------------------------
0: I enjoy walking with my cute dog," she said.

"I love walking with my dog," she added. "I love being with my dog. I love having a good time with my dog."

Read or Share this story:

1: I enjoy walking with my cute dog," she said.

"I love walking with my dog," she added. "I love being with my dog. I love having a good time with my dog."

2: I enjoy walking with my cute dog," she said.

"I love walking with my dog," she added. "I love being with my dog. I love having a good time with my dog."

She added: "I love

3: I enjoy walking with my cute dog," she said.

"I love walking with my dog," she added. "I love being with my dog. I love having a good time with my dog."

She added, "I love

4: I enjoy walking with my cute dog," she said.

"I love walking with my dog," she added. "I love being with my dog. I love having a good time with my dog."

She said she and her husband



## **Sampling**

В своей самой простой форме sampling означает случайный выбор следующего слова $w_t$ в соответствии с его условным распределением вероятностей:

$$w_t \sim P(w|w_{1:t-1})$$

Используя приведенный выше пример, на следующем рисунке визуализируется генерация языка при выборке.

![vanilla_sampling](https://raw.githubusercontent.com/patrickvonplaten/scientific_images/master/sampling_search.png)

Становится очевидным, что генерация языка с использованием sampling больше не является "детерминированной". Cлово
$\text{"car"}$ выбирается из условного распределения вероятностей $P(w |\text{"The"})$, за которым следует sampling $\text{"drives"}$ из $P(w |\text{"The"}, \text{"car"})$.

В `transformers` мы устанавливаем `do_sample=True` и деактивируем sampling *Top-K* через `top_k=0`. Не стесняйтесь изменять `random_seed`, чтобы поиграть с моделью.

In [9]:
# set seed to reproduce results. Feel free to change the seed though to get different results
torch.random.manual_seed(28)

# activate sampling and deactivate top_k by setting top_k sampling to 0
sample_output = model.generate(
    input_ids,
    do_sample=True,
    max_length=100,
    top_k=0
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog, Pesqua, while sitting at my desk. & simple yet super playful, he loves to walk amongst my furniture… He loves the experience of missing home and every dayIt's such a definite figure in my life! Far more enjoyable than traveling and so much more tranquil than going to work or school…the a complex and complex mindfulness that sees us walking more with capacity to not miss our work. He does not simply stare at everything to see if people or


Текст кажется нормальным, но при ближайшем рассмотрении оказывается, что он не очень связный. Слова *new hand sense* и *local batte harness* очень странные и звучат не так, как будто они были написаны человеком. Это большая проблема при выборке последовательностей слов: модели часто генерируют бессвязную тарабарщину.

Хитрость заключается в том, чтобы сделать распределение $P(w|w_{1:t-1})$ более четким (увеличивая вероятность слов с высокой вероятностью и уменьшая вероятность слов с низкой вероятностью), понизив так называемую "температуру" [softmax](https://en.wikipedia.org/wiki/Softmax_function#Smooth_arg_max ).

$q = \frac{exp(z_i / T)}{\sum_j exp(z_j / T)}$

Иллюстрация применения температуры к нашему примеру, приведенному выше, может выглядеть следующим образом.

![top_p_sampling](https://github.com/patrickvonplaten/scientific_images/blob/master/sampling_search_with_temp.png?raw=true)

Условное распределение следующего слова на шаге $t = 1$ становится намного более четким, практически не оставляя шансов для выбора слова $\text{"car"}$.


In [10]:
# set seed to reproduce results. Feel free to change the seed though to get different results
torch.random.manual_seed(0)

# use temperature to decrease the sensitivity to low probability candidates
sample_output = model.generate(
    input_ids,
    do_sample=True,
    max_length=50,
    top_k=0,
    temperature=1.5
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog along sidewalk seven between our stories... is "tree legs diffuse the spill that flows unfortunately wherever Carinky roams?"Plus I hear she so cagey. MTV

fb Countryines brood Sanskrit fever Symptoms Quarantine


## **Top-K Sampling**

[Fan et. al (2018)](https://arxiv.org/pdf/1805.04833.pdf ) ввел простую, но очень мощную схему выборки, называемую выборкой ***Top-K***. В выборке *Top-K* фильтруются *K* наиболее вероятных следующих слов, и масса вероятности перераспределяется только между этими *K* следующими словами.
GPT2 принял эту схему выборки, что стало одной из причин его успеха.

Мы расширили диапазон слов, используемых для обоих этапов выборки в приведенном выше примере, с 3 слов до 10 слов, чтобы лучше проиллюстрировать выборку *Top-K*.

![top_k_sampling](https://raw.githubusercontent.com/patrickvonplaten/scientific_images/master/top_k_sampling.png)

Установив $K = 6$, на обоих этапах выборки мы ограничиваем наш пул выборок 6 словами. В то время как 6 наиболее вероятных слов, определенных как $V_ {\text{top-K}}$, охватывают только *около* двух третей всей массы вероятности на первом шаге, они включают почти всю массу вероятности на втором шаге. Тем не менее, мы видим, что он успешно устраняет довольно странных кандидатов $\text{"not", "the", "small", "told"}$
на втором этапе отбора проб.


Давайте посмотрим, как *Top-K* можно использовать в библиотеке, установив `top_k=50`:

In [11]:
# set seed to reproduce results. Feel free to change the seed though to get different results
torch.random.manual_seed(0)

# set top_k to 50
sample_output = model.generate(
    input_ids,
    do_sample=True,
    max_length=100,
    top_k=20
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog, but it's like having your dog on your lap while the sun is up, and then being your dog and having him be like, 'You know what' in a way. I mean there was this one time we saw the train, but we never got into a fight. He's pretty much had it with that train. I have two other dogs and they just kind of went through each other's shit, and it wasn't like any other day because



Однако одна проблема с выборкой *Top-K* заключается в том, что она динамически не адаптирует количество слов, которые отфильтровываются из распределения вероятности следующего слова $P (w |w_ {1:t-1})$.
Это может быть проблематично, поскольку некоторые слова могут быть выбраны из очень четкого распределения (распределение справа на графике выше), в то время как другие - из гораздо более плоского распределения (распределение слева на графике выше).

На шаге $t=1$, *Top-K* исключает возможность
примеров $\text{"people", "big", "house", "cat"}$, которые кажутся разумными кандидатами. С другой стороны, на шаге $t= 2$ метод включает, возможно, неправильно подобранные слова $\text{"down", "a"}$ в выборку слов. Таким образом, ограничение пула выборок фиксированным размером *K* может привести к тому, что модель будет выдавать тарабарщину для четких распределений и ограничит творческий потенциал модели для плоского распределения.
Эта интуиция привела [Ari Holtzman et al. (2019)](https://arxiv.org/abs/1904.09751 ) для создания ***Top-p*** - или ***nucleus***-sampling.

## **Sampling Top-p (nucleus)**

Вместо выборки только из наиболее вероятных *K* слов, в выборке *Top-p* выбирается из наименьшего возможного набора слов, совокупная вероятность которых превышает вероятность *p*. Затем масса вероятности перераспределяется между этим набором слов. Таким образом, размер набора слов (количество слов в наборе) может динамически увеличиваться и уменьшаться в соответствии с распределением вероятности следующего слова. Ладно, это было очень многословно, давайте визуализируем.

![top_p_sampling](https://github.com/patrickvonplaten/scientific_images/blob/master/top_p_sampling.png?raw=true)

Установив $p=0.92$, выборка *Top-p* выбирает *минимальное* количество слов, которое в совокупности должно превышать $p=92\%$ от массы вероятности, определяемой как $V_{\text{top-p}}$. В первом примере это включало 9 наиболее вероятных слов, тогда как во втором примере нужно выбрать только 3 лучших слова, чтобы их вероятностная масса превысила 92%. На самом деле все очень просто! Можно видеть, что стратегия сохраняет широкий диапазон слов, где следующее слово, возможно, менее предсказуемо, *например* $P(w | \text{"The"})$, и только несколько слов, когда следующее слово кажется более предсказуемым, *например* $P(w | \text{"The", "car"})$.

Мы активируем выборку *Top-p*, установив `0 < top_p < 1`:

In [12]:
# set seed to reproduce results. Feel free to change the seed though to get different results
torch.random.manual_seed(3)

# deactivate top_k sampling and sample only from 92% most likely words
sample_output = model.generate(
    input_ids,
    do_sample=True,
    max_length=100,
    top_p=0.80,
    top_k=0
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog.

Meet Szyphos:

I can't believe we're here. We'd like to introduce you to Szyphos. I just got married. She was just two months old and my adorable little sizzler and we're thrilled to have met so many beautiful, sweet, kind, sweet and loving women here at Pink Beach. She's a REAL mom, and I have been coming up with names for my younger sister, Will


Отлично, звучит так, словно это мог бы написать человек. Ну, может быть, еще не совсем.

Хотя в теории *Top-p* кажется более элегантным, чем *Top-K*, оба метода хорошо работают на практике. *Top-p* также можно использовать в сочетании с *Top-K*, что позволяет избежать слов с очень низким скором, обеспечивая при этом некоторый динамический выбор.

Наконец, чтобы получить несколько независимо отобранных выходных данных, мы можем установить параметр `num_return_sequences > 1`:

In [13]:
# set seed to reproduce results. Feel free to change the seed though to get different results
torch.random.manual_seed(0)

# set top_k = 50 and set top_p = 0.95 and num_return_sequences = 3
sample_outputs = model.generate(
    input_ids,
    do_sample=True,
    max_length=100,
    top_k=20,
    top_p=0.80,
    num_return_sequences=3
)

print("Output:\n" + 100 * '-')
for i, sample_output in enumerate(sample_outputs):
  print("{}: {}".format(i, tokenizer.decode(sample_output, skip_special_tokens=True)))

Output:
----------------------------------------------------------------------------------------------------
0: I enjoy walking with my cute dog, and she is the perfect companion for my dogs. She loves to run and walk with me, and she is a wonderful person who is always smiling and smiling with us. She also enjoys the way she's always been able to give us her own unique smile.

I have a small dog named "Cathy" who loves to walk, and she has a sweet and loving personality. I have also got a few puppies called "Cynthia," named
1: I enjoy walking with my cute dog, but my best friend, who is also a dog lover, was so kind. When she asked me to give her some treats to eat, I had her give her a cookie. She said, "Thank you very much for giving me a cookie, my dog is just fine." She said, "I'm so happy. My dog loves it." It was a good experience!

This is my first visit and I really enjoyed it. My dogs love
2: I enjoy walking with my cute dog, and we are both very interested in how he is doing

## Constrained generation

Для лучшей читаемости мы будем использовать end-of-line в качестве токена EOS, вместо того, чтобы просто всегда генерировать 100 токенов.

In [14]:
END_OF_LINE = tokenizer('\n').input_ids[0]
print(END_OF_LINE)

198


Поиск по базовому лучу выдает очень похожие предложения; все они содержат "not sure" или "don't think".

In [15]:
beam_outputs = model.generate(
    input_ids,
    max_length=500,
    num_beams=5,
    no_repeat_ngram_size=2,
    num_return_sequences=5,
    early_stopping=True,
    eos_token_id=END_OF_LINE,
)

print("Output:\n" + 100 * '-')
for i, beam_output in enumerate(beam_outputs):
  print("{}: {}".format(i, tokenizer.decode(beam_output, skip_special_tokens=True)))

Output:
----------------------------------------------------------------------------------------------------
0: I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again.

1: I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with her again.

2: I enjoy walking with my cute dog, but I don't think I'll ever be able to walk with her again.

3: I enjoy walking with my cute dog, but I don't think I'll ever be able to walk with him again.

4: I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again. I don't know what to do.



Что произойдет, если мы запретим модели использовать эти фразы?



In [16]:
beam_outputs = model.generate(
    input_ids,
    max_length=50,
    num_beams=5,
    no_repeat_ngram_size=2,
    num_return_sequences=5,
    early_stopping=True,
    eos_token_id=END_OF_LINE,
    bad_words_ids=tokenizer([' sure', ' want', ' like', 'able'])['input_ids'],
)

print("Output:\n" + 100 * '-')
for i, beam_output in enumerate(beam_outputs):
  print("{}: {}".format(i, tokenizer.decode(beam_output, skip_special_tokens=True)))

Output:
----------------------------------------------------------------------------------------------------
0: I enjoy walking with my cute dog, but I don't know if I'll ever be able to walk with him again."

1: I enjoy walking with my cute dog, but I don't know if I'll ever be able to walk with him again.

2: I enjoy walking with my cute dog, but I don't know if I'll ever be able to walk with her again."

3: I enjoy walking with my cute dog, but I don't think I'll ever be able to walk with her again.

4: I enjoy walking with my cute dog, but I don't know if I'll ever be able to walk with him again. He's a great dog.



In [17]:
tokenizer(['sure', ' sure', ' I am not sure'])['input_ids']

[[19532], [1654], [314, 716, 407, 1654]]

Мы видим, что смысл этих текстов сильно изменился – но каким-то непредсказуемым образом.

Можем ли мы заставить модель написать текст с участием "cat"?

In [18]:
beam_outputs = model.generate(
    input_ids,
    max_length=100,
    num_beams=8,
    no_repeat_ngram_size=2,
    num_return_sequences=5,
    early_stopping=True,
    eos_token_id=END_OF_LINE,
    trust_remote_code=True,
    bad_words_ids=tokenizer([' sure', ' think', " but I'm not a", ' lover'])['input_ids'],
    force_words_ids=[tokenizer([' cat','cat'], add_special_tokens=False).input_ids],
)

print("Output:\n" + 100 * '-')
for i, beam_output in enumerate(beam_outputs):
  print("{}: {}".format(i, tokenizer.decode(beam_output, skip_special_tokens=True)))

[transformers] Constrained Beam Search was moved to a `custom_generate` repo: https://hf.co/transformers-community/constrained-beam-search. To prevent loss of backward compatibility, add `custom_generate='transformers-community/constrained-beam-search'` to your `generate` call before v4.62.0.


generate.py:   0%|          | 0.00/18.2k [00:00<?, ?B/s]

beam_constraints.py:   0%|          | 0.00/19.3k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/transformers-community/constrained-beam-search:
- custom_generate/beam_constraints.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


beam_search.py:   0%|          | 0.00/34.9k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/transformers-community/constrained-beam-search:
- custom_generate/beam_search.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/transformers-community/constrained-beam-search:
- custom_generate/generate.py
- beam_constraints.py
- beam_search.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'force_words_ids', 'no_repeat_ngram_size', 'early_stopping', 'num_beams', 'eos_token_id', 'num_return_sequences', 'bad_words_ids'}) is deprecated and will be removed in future versions. Please pass either a `ge

Output:
----------------------------------------------------------------------------------------------------
0: I enjoy walking with my cute dog, but I don't like to walk alone. So I decided to do something about it. I started by taking a picture of my dog and then I took a photo of myself with her. Then I went back to my car and took another picture. It took about 10 minutes to get to the point where I was able to take the picture and post it on my Facebook page. The next day, I got a call from a friend of mine who told me
1: I enjoy walking with my cute dog, but I don't like to walk alone. So I decided to do something about it. I started by taking a picture of my dog and then I took a photo of myself with her. Then I went back to my car and took another picture. It took about 10 minutes to get to the point where I was able to take the picture and post it on my Facebook page. The next day, I got a call from a friend who said, "Hey
2: I enjoy walking with my cute dog, but I don't like 

Уточнение: **force_words_ids** - это список ограничений. Каждое ограничение представляет собой список выражений, таких, что по крайней мере одно выражение должно быть включено в сгенерированный текст. И каждое выражение - это просто список токенов.

Смотрите обсуждение в [the HF pull request](https://github.com/huggingface/transformers/issues/14081 ), или прочтите статью "[Guided Generation of Cause and Effect](https://www.ijcai.org/proceedings/2020/0502.pdf )" Ли и др., где был предложен алгоритм.  

Чтобы оценить силу этих ограничений, давайте заставим модель включить мышь (или даже много мышей) в текст. Мы также можем ослабить ограничение "кошка", разрешив вместо него использовать слова "кошки", "котенок" или "кошачий".

In [19]:
beam_outputs = model.generate(
    input_ids,
    max_length=50,
    num_beams=5,
    top_p = 0.90,
    no_repeat_ngram_size=2,
    num_return_sequences=5,
    early_stopping=True,
    eos_token_id=END_OF_LINE,
    trust_remote_code=True,
    bad_words_ids=tokenizer(['sure', 'think'], add_prefix_space=True)['input_ids'],
    force_words_ids = [
        tokenizer(['cat', 'cats', 'kitten', 'feline', 'Cat', 'Cats'], add_prefix_space=True, add_special_tokens=False).input_ids,
        tokenizer(['mouse', 'mice'], add_prefix_space=True, add_special_tokens=False).input_ids,
    ],
)

print("Output:\n" + 100 * '-')
for i, beam_output in enumerate(beam_outputs):
  print("{}: {}".format(i, tokenizer.decode(beam_output, skip_special_tokens=True)))

[transformers] The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'force_words_ids', 'no_repeat_ngram_size', 'top_p', 'early_stopping', 'num_beams', 'eos_token_id', 'num_return_sequences', 'bad_words_ids'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Output:
----------------------------------------------------------------------------------------------------
0: I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again. I've been trying to get him to stop, and I don't know if he'll be happy. He's been so
1: I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again. I've been trying to get him to stop, and I don't know if he'll be happy. He's beenf
2: I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again. I've been trying to get him to stop, and I don't know if he'll be happy. He'skitten
3: I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again. I've been trying to get him to stop, and I don't know if he'll be happy.Catsmice
4: I enjoy walking with my cute dog, but I'm not sure if I'll ever be able to walk with him again. I've been trying to get him to stop, and I don't know if he'l

## **Приложение**

Есть пара дополнительных параметров для метода `generate`, которые не были упомянуты выше. Мы кратко объясним их здесь!

- `min_length` можно использовать, чтобы заставить модель не выдавать токен EOS (= не заканчивать предложение) до достижения `min_length`. Это довольно часто используется при суммаризации, но может быть полезно в целом, если пользователь хочет получить более длинные выходные данные.
- `repetition_penalty` может использоваться для "наказания" слов, которые уже были сгенерированы или принадлежат контексту. Впервые он был представлен [Kesker et al. (2019)](https://arxiv.org/abs/1909.05858 ) и также используется в цели обучения в [Welleck et al. (2019)](https://arxiv.org/pdf/1908.04319.pdf ). Это может быть довольно эффективно для предотвращения повторений, но, по-видимому, очень чувствительно к различным моделям и вариантам использования, *например* смотрите это [обсуждение](https://github.com/huggingface/transformers/pull/2303 ) на Github.

- `attention_mask` можно использовать для маскировки дополненных токенов
- `pad_token_id`, `bos_token_id`, `eos_token_id`: Если в модели по умолчанию нет этих токенов, пользователь может вручную выбрать другие идентификаторы токенов для их представления.


In [20]:
beam_outputs = model.generate(
    input_ids,
    max_length=50,
    num_beams=5,
    no_repeat_ngram_size=2,
    num_return_sequences=5,
    early_stopping=True,
    eos_token_id=END_OF_LINE,
    trust_remote_code=True,
    bad_words_ids=tokenizer([' sure', ' think'])['input_ids'],
    force_words_ids = [
        tokenizer([' cat', ' cats', ' kitten', ' feline', ' Cat', ' Cats'], add_special_tokens=False).input_ids,
        tokenizer([' mouse', ' mice'], add_special_tokens=False).input_ids,
    ],
    repetition_penalty=2.0
)

print("Output:\n" + 100 * '-')
for i, beam_output in enumerate(beam_outputs):
  print("{}: {}".format(i, tokenizer.decode(beam_output, skip_special_tokens=True)))

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'force_words_ids', 'no_repeat_ngram_size', 'early_stopping', 'num_beams', 'repetition_penalty', 'eos_token_id', 'num_return_sequences', 'bad_words_ids'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.


Output:
----------------------------------------------------------------------------------------------------
0: I enjoy walking with my cute dog. I love to play with her and she loves to be around me. She's a great companion for me."The UESPWiki – Your source for The Elder Scrolls since 1995

, by Jage
1: I enjoy walking with my cute dog. I love to play with her and she loves to be around me. She's a great companion for me."The UESPWiki – Your source for The Elder Scrolls since 1995

, by feline
2: I enjoy walking with my cute dog. I love to play with her and she loves to be around me. She's a great companion for me."The UESPWiki – Your source for The Elder Scrolls since 1995

, by J f
3: I enjoy walking with my cute dog. I love to play with her and she loves to be around me. She's a great companion for me."The UESPWiki – Your source for The Elder Scrolls since 1995

, by Cat mouse
4: I enjoy walking with my cute dog. I love to play with her and she loves to be around me. She's a great